# 06 — Station Value Extraction

Extracts aligned raster values at gauge station coordinates.

In [ ]:
from pathlib import Path
import sys
import warnings

warnings.filterwarnings("ignore")

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data").exists() and (candidate / "configs").exists():
            return candidate
    raise FileNotFoundError(
        "Project root was not found. Run this notebook from inside the repository."
    )

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
CONFIG_DIR = PROJECT_ROOT / "configs"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
MODEL_DIR = PROJECT_ROOT / "models"

for folder in [INTERIM_DIR, PROCESSED_DIR, OUTPUT_DIR, MODEL_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
import pandas as pd
import rasterio
from pyproj import Transformer

gauge_path = PROCESSED_DIR / "station_samples" / "gauge_monthly_clean.csv"
gauge = pd.read_csv(gauge_path)

required = {"latitude", "longitude"}
if not required.issubset(gauge.columns):
    raise ValueError("Gauge CSV must contain latitude and longitude columns.")

aligned_files = sorted((PROCESSED_DIR / "aligned_rasters").rglob("*.tif"))
if not aligned_files:
    raise FileNotFoundError("Aligned rasters are missing. Run Notebook 05 first.")

In [ ]:
samples = gauge.copy()

for raster_path in aligned_files:
    column_name = f"{raster_path.parent.name}_{raster_path.stem}"

    with rasterio.open(raster_path) as src:
        transformer = Transformer.from_crs(
            "EPSG:4326", src.crs, always_xy=True
        )
        xy = [
            transformer.transform(lon, lat)
            for lon, lat in zip(samples["longitude"], samples["latitude"])
        ]
        values = [value[0] for value in src.sample(xy)]
        samples[column_name] = values

output_path = PROCESSED_DIR / "station_samples" / "station_raster_samples.csv"
samples.to_csv(output_path, index=False)
display(samples.head())
print(f"Saved: {output_path}")